# 02 - Sentiment Analysis
**QSS 20 Final Project | Olivia Tak**

This notebook runs VADER sentiment analysis on video titles to compute 
arousal scores and sentiment labels, then checks distributions and correlations.

## Imports

In [1]:
import sys
!{sys.executable} -m pip install vaderSentiment


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy import stats

## Load cleaned data

In [3]:
# load the cleaned dataset from previous notebook
df = pd.read_csv('../data/df_2023_clean.csv')
print(df.shape)

(12176, 21)


## Run VADER on video titles

In [4]:
# initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# apply VADER to each title and extract compound score
df['compound'] = df['title'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

# arousal score = absolute value of compound (high positive OR negative = high arousal)
df['arousal_score'] = df['compound'].abs()

# sentiment label based on compound score thresholds
df['sentiment_label'] = df['compound'].apply(lambda x: 'positive' if x > 0.05 
                                              else ('negative' if x < -0.05 
                                              else 'neutral'))

print(df[['title', 'compound', 'arousal_score', 'sentiment_label']].head(10))

                                               title  compound  arousal_score  \
0  Peach Bowl: Ohio State Buckeyes vs. Georgia Bu...    0.0000         0.0000   
1                              Tom MacDonald - Ghost   -0.3182         0.3182   
2  Ñengo Flow, Bad Bunny - Gato de Noche ( Video ...   -0.5423         0.5423   
3  Seattle weather conditions: Ongoing power outa...    0.0000         0.0000   
4         Honda’s $225 Million Mistake – Rune Review   -0.3400         0.3400   
5                         i became an italian farmer    0.0000         0.0000   
6             What Dan and Phil Text Each Other 2022    0.0000         0.0000   
7  It’s Beginning To Look A Lot Like Christmas (c...    0.3612         0.3612   
8        That '90s Show | Official Trailer | Netflix    0.0000         0.0000   
9          20 Things You Didn't Know About The NBA..    0.0000         0.0000   

  sentiment_label  
0         neutral  
1        negative  
2        negative  
3         neutral  
4       

## Check distributions

In [6]:
# check distribution of sentiment labels
print(df['sentiment_label'].value_counts())
print()
print(df['arousal_score'].describe().round(3))

sentiment_label
neutral     6686
positive    2775
negative    2715
Name: count, dtype: int64

count    12176.000
mean         0.209
std          0.264
min          0.000
25%          0.000
50%          0.000
75%          0.440
max          0.954
Name: arousal_score, dtype: float64


In [7]:
# mean views by sentiment group
print(df.groupby('sentiment_label')['view_count'].mean().round(0))

sentiment_label
negative    1158660.0
neutral     1416410.0
positive    1114614.0
Name: view_count, dtype: float64


## Summary statistics by sentiment group

In [9]:
# mean view count and arousal score grouped by sentiment label
summary = (df.groupby('sentiment_label')
             .agg({'view_count': 'mean', 
                   'arousal_score': 'mean',
                   'video_id': 'count'})
             .rename(columns={'video_id': 'count'})
             .round(2))

print(summary)

                 view_count  arousal_score  count
sentiment_label                                  
negative         1158660.16           0.46   2715
neutral          1416409.62           0.00   6686
positive         1114614.29           0.46   2775


## Save data with sentiment scores

In [10]:
# save dataframe with sentiment scores for use in visualization notebook
df.to_csv('../data/df_2023_sentiment.csv', index=False)
print("Saved successfully")

Saved successfully
